# Ecommerce Customer Clustering
**Assignment checklist:** data understanding -> missing values -> encoding -> outliers -> feature selection -> scaling -> elbow -> silhouette -> K-Means -> profiling -> business meaning

Pink sklearn warning about threads is harmless. Ignore it.

Run top to bottom. CSV must sit next to this notebook.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option('display.max_columns', 30)
df = pd.read_csv('ecommerce_customer_clustering_12000.csv')
print(df.shape)
df.head()


## 1. Data understanding


In [ ]:
print(df.info())
print(df.isnull().sum())
print('Duplicates:', df.duplicated().sum())
print('Unique IDs:', df['Customer_ID'].nunique())
df.describe().T


In [ ]:
cat_cols = ['City', 'Membership_Tier', 'Device_Type', 'Preferred_Payment', 'Favorite_Category']
for c in cat_cols:
    print('===', c, '===')
    print(df[c].value_counts())


## 2. Missing-value treatment
Fill numeric holes with the median.


In [ ]:
num_missing = ['Annual_Income','App_Visits_Per_Month','Orders_Per_Month','Average_Order_Value','Discount_Usage_Percentage','Average_Rating']
for col in num_missing:
    df[col] = df[col].fillna(df[col].median())
print(df.isnull().sum())


## 3. Categorical encoding
Membership_Tier is ordered. Other labels are one-hot encoded for the rubric.


In [ ]:
tier_map = {'Basic': 1, 'Silver': 2, 'Gold': 3, 'Platinum': 4}
df['Tier_Ordinal'] = df['Membership_Tier'].map(tier_map)
dummies = pd.get_dummies(df[['Device_Type', 'Preferred_Payment', 'Favorite_Category']], drop_first=True)
print(df['Tier_Ordinal'].value_counts().sort_index())
print(list(dummies.columns))


## 4. Outlier analysis (IQR)
lower = Q1 - 1.5*IQR, upper = Q3 + 1.5*IQR. Cap instead of deleting rows.


In [ ]:
def iqr_report(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = ((s < lower) | (s > upper)).sum()
    return pd.Series({'Q1': q1, 'Q3': q3, 'IQR': iqr, 'lower': lower, 'upper': upper, 'n_outliers': n})
check_cols = ['Annual_Income','Average_Order_Value','Annual_Spend','Support_Tickets','Days_Since_Last_Order','Engagement_Score']
print(df[check_cols].apply(iqr_report).T.round(2))
print('Negative engagement:', (df['Engagement_Score'] < 0).sum())


In [ ]:
df['Engagement_Score'] = df['Engagement_Score'].clip(lower=0)
cap_cols = ['Annual_Income','Average_Order_Value','Annual_Spend','Support_Tickets']
for col in cap_cols:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    df[col] = df[col].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)


## 5-6. Feature selection and scaling
Drop Customer_ID. Scale before K-Means.


In [ ]:
features = ['Age','Annual_Income','Tenure_Months','App_Visits_Per_Month','Orders_Per_Month','Average_Order_Value','Annual_Spend','Discount_Usage_Percentage','Returns_Count','Days_Since_Last_Order','Engagement_Score']
X_raw = df[features].copy()
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)
print(X.shape)


## 7-8. Elbow and silhouette


In [ ]:
ks = range(2, 9)
inertias, sils = [], []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km.fit_predict(X)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X, labels_k))
    print(f'K={k} inertia={km.inertia_:.0f} silhouette={sils[-1]:.3f}')
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(list(ks), inertias, marker='o')
ax[0].set_title('Elbow (inertia)')
ax[1].plot(list(ks), sils, marker='o', color='green')
ax[1].set_title('Silhouette')
plt.tight_layout()
plt.show()


## 9. K-Means
Label numbers have no meaning until you profile them.


In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans.fit(X)
df['Cluster'] = kmeans.labels_
print(df['Cluster'].value_counts().sort_index())
print('silhouette:', silhouette_score(X, kmeans.labels_))


## 10-11. Profile and business meaning


In [ ]:
profile = df.groupby('Cluster')[features].mean().round(2)
profile['n_customers'] = df.groupby('Cluster').size()
print(profile.T)
print(pd.crosstab(df['Cluster'], df['Membership_Tier'], normalize='index').round(2))
